In [0]:
%run ./common_utils

In [0]:
import time
CAT = 'machine_health.prod'
AUDIT = CAT + '.pipeline_run_audit'
declare_widgets(merge_delay_sec=0)
mt = dbutils.widgets.get('machine_type')
delay = int(dbutils.widgets.get('merge_delay_sec') or 0)
TARGET = f'{CAT}.health_decisions'
with audited() as a:
    src = (spark.table(f'{CAT}.predictions_latest').where(scope(mt))
           .select('sensor_id', 'machine_type', 'reading_time', 'axis', 'is_healthy', 'model_tag')
           .withColumn('decision_note', F.lit('rule_flagged')))
    if delay > 0:
        slow = F.udf(lambda v: (time.sleep(delay), v)[1], 'string')
        src = src.withColumn('sensor_id', slow('sensor_id'))
    src.createOrReplaceTempView('src_view')
    a.update(table_name=TARGET, operation='MERGE',
             read_version=spark.sql(f'DESCRIBE HISTORY {TARGET} LIMIT 1').first().version)
    try:
        spark.sql(f'''
          MERGE INTO {TARGET} t USING src_view s
          ON t.sensor_id = s.sensor_id AND t.reading_time = s.reading_time AND t.axis = s.axis
          WHEN MATCHED THEN UPDATE SET *
          WHEN NOT MATCHED THEN INSERT *''')
    except Exception:
        rv = a['read_version']
        h = (spark.sql(f'DESCRIBE HISTORY {TARGET}').where(f'version > {rv}')
             .orderBy('version').first())
        if h is not None:
            a.update(conflicting_version=h.version,
                     conflicting_job_run_id=str(h.job.runId) if h.job else None)
        raise
    unhealthy = (
        spark.table(TARGET)
        .where(scope(mt))
        .where('is_healthy = 0')
    )
    unhealthy.createOrReplaceTempView('unhealthy_view')
    spark.sql(f'''
        MERGE INTO {CAT}.alert_candidates t
        USING unhealthy_view s
        ON t.sensor_id = s.sensor_id 
        AND t.reading_time = s.reading_time 
        AND t.axis = s.axis
        WHEN NOT MATCHED THEN INSERT (
            sensor_id, machine_type, reading_time, axis, is_healthy, model_tag, decision_note
        )
        VALUES (
            s.sensor_id, s.machine_type, s.reading_time, s.axis, s.is_healthy, s.model_tag, s.decision_note
        )
    ''')


/home/spark-546823a3-d494-4611-bd7e-ca/.ipykernel/63/command-7089605854406250-1338218180:94: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_time=datetime.utcnow()
/home/spark-546823a3-d494-4611-bd7e-ca/.ipykernel/63/command-7089605854406250-1338218180:108: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  rec['end_time'] = datetime.utcnow()
